# Ensemble Analysis — Technique 0 (Average-then-Analyze)

In plasma UQ studies it is common to run the same simulation multiple times with different random seeds (or slightly perturbed initial conditions) to obtain an ensemble of realisations.  `quends` represents this as an `Ensemble` of `DataStream` objects and provides three distinct statistical aggregation techniques.

## Technique 0 — Average-then-Analyze

**Procedure**: All ensemble members are concatenated into a single combined time series, and statistics (mean, standard error, confidence interval) are computed on that pooled stream as if it were one long run.

**When to use**:
- Members are statistically exchangeable (same physics, different seeds).
- You want maximum data pooling and a single global estimate.
- You do *not* need member-specific estimates.

**Strengths**: Simple, maximises use of available data.  
**Caveats**: If members have different lengths or different start offsets after trimming, concatenation may dilute the effective sample size estimate.

Compare with:
- **Technique 1** (`04_technique1_ensemble.ipynb`): Block-average each member, then pool the blocks.
- **Technique 2** (`05_technique2_ensemble.ipynb`): Inverse-variance weighted mean across members.

In [ ]:
import quends as qnds

print(f"quends version: {qnds.__version__}")

## 1. Load the Ensemble

Each ensemble member is a separate CSV file.  We load them individually with `qnds.from_csv()` and pass the list to `qnds.Ensemble()`.

In [ ]:
files = [
    "gx/ensemble/tprim_2_5_a.out.csv",
    "gx/ensemble/tprim_2_5_b.out.csv",
    "gx/ensemble/tprim_2_5_c.out.csv",
]

members = [qnds.from_csv(f) for f in files]
ens = qnds.Ensemble(members)

print(f"Ensemble has {len(ens.data_streams)} members")
for i, ds in enumerate(ens.data_streams):
    print(f"  Member {i}: {len(ds)} time steps, variables = {ds.variables()}")

## 2. Visualise the Raw Ensemble

Before trimming it is useful to inspect each member's trace to get a feel for the transient length and the run-to-run variability.

In [ ]:
%matplotlib inline

plotter = qnds.Plotter()

for i, ds in enumerate(ens.data_streams):
    print(f"--- Member {i} ---")
    plotter.trace_plot(ds, ["HeatFlux_st"])

## 3. Trim the Ensemble

`Ensemble.trim()` is the **public Ensemble-level trim API**.  It applies a trim strategy to every member independently and returns a new `Ensemble` containing only the surviving (trimmed) members.

Internally, `Ensemble.trim()` calls `build_trim_strategy(method, window_size, ...)` from `quends.base.trim` to construct the strategy object, then wraps it in `TrimDataStreamOperation` and applies it per member.  The `method` string corresponds to the factory keys shown in notebook 01.

For users who need finer control (e.g. different strategies per member, or inspection of the strategy object before applying it), the low-level API can be used directly:

```python
from quends.base.trim import QuantileTrimStrategy, TrimDataStreamOperation
strategy = QuantileTrimStrategy(window_size=20, robust=True)
op = TrimDataStreamOperation(strategy=strategy)
trimmed_member = op(member_ds, column_name="HeatFlux_st")
```

In [ ]:
# Trim all members using the "std" (QuantileTrim) strategy
trimmed_ens = ens.trim("HeatFlux_st", method="std", window_size=20)

print(f"Original ensemble : {len(ens.data_streams)} members")
print(f"Trimmed ensemble  : {len(trimmed_ens.data_streams)} members survived")

for i, ds in enumerate(trimmed_ens.data_streams):
    print(f"  Member {i}: {len(ds)} steps remaining  (t_start = {ds.data['time'].iloc[0]:.3f})")

### Visualise the Trimmed Members

Confirm that the trim boundaries look sensible for each member.

In [ ]:
for i, ds in enumerate(trimmed_ens.data_streams):
    print(f"--- Trimmed Member {i} ---")
    plotter.trace_plot(ds, ["HeatFlux_st"])

## 4. Compute Ensemble Statistics — Technique 0

`Ensemble.compute_statistics(col, technique=0)` concatenates all trimmed member streams into one pooled series and computes the mean and confidence interval on that combined stream.

The `technique` argument selects the aggregation method:

| Value | Name | Description |
|-------|------|-------------|
| `0` | Average-then-Analyze | Pool all members into one stream, compute statistics once |
| `1` | Pooled-Block | Block-average each member, pool the blocks |
| `2` | Member-wise IVW | Inverse-variance weighted mean across members |

In [ ]:
results = trimmed_ens.compute_statistics("HeatFlux_st", technique=0)
print("Technique 0 results:")
print(results)

## 5. Display Results

`qnds.Exporter()` formats the statistics output for human reading or downstream consumption.

In [ ]:
exporter = qnds.Exporter()

# Tabular display
exporter.display_dataframe(results)

In [ ]:
# JSON display (useful for saving to file or passing to downstream scripts)
exporter.display_json(results)

## 6. Quick Sanity Check — Member-wise Means

Before trusting the ensemble result it is good practice to verify that each member's individual mean is consistent with the ensemble estimate.  Large outliers may indicate a member that failed to reach steady state.

In [ ]:
print("Member-wise means for HeatFlux_st:")
for i, ds in enumerate(trimmed_ens.data_streams):
    m = ds.mean()
    ci = ds.confidence_interval()
    print(f"  Member {i}: mean = {m},  95% CI = {ci}")

## Summary

| Step | API |
|------|-----|
| Load members | `qnds.from_csv(path)` (one per member) |
| Build ensemble | `qnds.Ensemble(list_of_datastreams)` |
| Trim ensemble | `ens.trim(col, method="std", window_size=20)` |
| Compute stats | `trimmed_ens.compute_statistics(col, technique=0)` |
| Display | `qnds.Exporter().display_dataframe(results)` |

**Note on the low-level API**: `Ensemble.trim()` is the recommended interface for ensemble-level trimming.  If you need explicit control over the strategy object (e.g. to inspect trim indices or pass non-default parameters not exposed through `Ensemble.trim()`), use `TrimDataStreamOperation` directly as shown in notebook 02.

**Next**: `04_technique1_ensemble.ipynb` — Technique 1 (Pooled-Block averaging).